In [1]:
# Cell 1 — imports and connection
import os
import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
from collections import Counter
import json
 
load_dotenv()
 
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database='ANALYTICS_PROD',
    schema='PUBLIC',
)
 
def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)
 
print("Connected.")

Connected.


In [2]:
# ── Cell 2 — load mart ────────────────────────────────────────────────────────
df = run_query("SELECT * FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS")
df.columns = [c.lower() for c in df.columns]

# Parse arrays
def parse_arr(val):
    if isinstance(val, list): return val
    if isinstance(val, str):
        try: return json.loads(val)
        except: return []
    return []

for col in ["tech_stack_required", "tech_stack_preferred", "paradigms_required", "paradigms_preferred"]:
    df[col] = df[col].apply(parse_arr)

# Parse dates
for col in ["date_posted", "ingested_at", "enriched_at"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Numeric
for col in ["final_salary_min", "final_salary_max", "years_required_min", "years_required_max", "confidence_score"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Booleans
for col in ["acknowledges_ai", "explicitly_encourages_applicants"]:
    df[col] = df[col].apply(lambda x: True if str(x).strip().upper() in ("TRUE", "1", "YES") else False)

# early_career_tier comes straight from the mart now — no notebook-side computation needed
EARLY_CAREER_ORDER = ["entry_or_junior", "mid"]

# Role grouping: title_role_bucket (regex-classified from job_title) replaces
# ingestion_query (which search term surfaced the posting) as the role axis.
# ingestion_query was found to mislabel ~36% of Analytics Engineer postings as
# Data Engineer due to JSearch's loose topical search matching — see PROJECT.md.
# 'no_match' rows (titles that didn't cleanly map to one of the four target
# roles) are excluded here since every role-grouped stat below needs a clean
# four-way split — they remain in the mart, visible on Pipeline Health.
n_before_match_filter = len(df)
df = df[df["title_role_bucket"] != "no_match"].copy()
n_no_match_excluded = n_before_match_filter - len(df)

# Salary mid
salary_df = df.dropna(subset=["final_salary_min", "final_salary_max"]).copy()
salary_df["salary_mid"] = (salary_df["final_salary_min"] + salary_df["final_salary_max"]) / 2

print(f"Loaded {n_before_match_filter} postings ({n_no_match_excluded} excluded as no_match title) → {len(df)} analyzed")

Loaded 485 postings (37 excluded as no_match title) → 448 analyzed


In [3]:
# ── Cell 3 — top level snapshot ───────────────────────────────────────────────
print("=== TOP LEVEL ===")
print(f"Total postings:       {len(df)}")
print(f"Unique companies:     {df['company_name'].nunique()}")
print(f"Role types:           {df['title_role_bucket'].nunique()}")
print(f"Sources:              {df['source'].nunique()}")
print(f"Salary disclosed:     {len(salary_df)} ({len(salary_df)/len(df):.0%})")
print(f"LLM enriched:         {df['role_archetype'].notna().sum()} ({df['role_archetype'].notna().mean():.0%})")
print(f"Date range:           {df['date_posted'].min().date()} → {df['date_posted'].max().date()}")
print(f"Last ingested:        {df['ingested_at'].max().date()}")
 

=== TOP LEVEL ===
Total postings:       448
Unique companies:     370
Role types:           4
Sources:              3
Salary disclosed:     269 (60%)
LLM enriched:         448 (100%)
Date range:           2026-05-27 → 2026-06-25
Last ingested:        2026-06-25


In [4]:
# ── Cell 4 — postings by role type ───────────────────────────────────────────
print("\n=== POSTINGS BY ROLE TYPE ===")
print(df["title_role_bucket"].value_counts().to_string())


=== POSTINGS BY ROLE TYPE ===
title_role_bucket
Data Analyst          196
Data Scientist        115
Data Engineer         111
Analytics Engineer     26


In [5]:
# ── Cell 5 — postings by source ──────────────────────────────────────────────
print("\n=== POSTINGS BY SOURCE ===")
print(df["source"].value_counts().to_string())
 
print("\n=== POSTINGS BY ROLE × SOURCE ===")
print(df.groupby(["title_role_bucket", "source"]).size().unstack(fill_value=0).to_string())


=== POSTINGS BY SOURCE ===
source
theirstack    168
jsearch       144
builtin       136

=== POSTINGS BY ROLE × SOURCE ===
source              builtin  jsearch  theirstack
title_role_bucket                               
Analytics Engineer        5       10          11
Data Analyst             57       67          72
Data Engineer            32       39          40
Data Scientist           42       28          45


In [6]:
# ── Cell 6 — work model ───────────────────────────────────────────────────────
print("\n=== WORK MODEL (overall) ===")
print(df["work_model"].value_counts().to_string())
 
print("\n=== WORK MODEL by ROLE ===")
print(df.groupby(["title_role_bucket", "work_model"]).size().unstack(fill_value=0).to_string())


=== WORK MODEL (overall) ===
work_model
onsite    263
remote    130
hybrid     55

=== WORK MODEL by ROLE ===
work_model          hybrid  onsite  remote
title_role_bucket                         
Analytics Engineer       5      15       6
Data Analyst            22     123      51
Data Engineer           16      59      36
Data Scientist          12      66      37


In [7]:
# ── Cell 7 — early-career tier distribution ──────────────────────────────────
print("\n=== EARLY CAREER TIER (overall) ===")
print(df["early_career_tier"].value_counts().to_string())

print("\n=== EARLY CAREER TIER by ROLE ===")
print(df.groupby(["title_role_bucket", "early_career_tier"]).size().unstack(fill_value=0).to_string())


=== EARLY CAREER TIER (overall) ===
early_career_tier
mid                225
entry_or_junior     68

=== EARLY CAREER TIER by ROLE ===
early_career_tier   entry_or_junior  mid
title_role_bucket                       
Analytics Engineer                1   15
Data Analyst                     32   86
Data Engineer                    15   57
Data Scientist                   20   67


In [8]:
# ── Cell 8 — salary by role ───────────────────────────────────────────────────
print("\n=== SALARY by ROLE (median, where disclosed) ===")
sal_by_role = (
    salary_df.groupby("title_role_bucket")["salary_mid"]
    .agg(["median", "count", "min", "max", "std"])
    .round(0)
)
sal_by_role.columns = ["median", "n", "min", "max", "std"]
print(sal_by_role.to_string())

print("\n=== SALARY by ROLE × EARLY CAREER TIER ===")
sal_tier = salary_df[salary_df["early_career_tier"].isin(EARLY_CAREER_ORDER)]
print(
    sal_tier.groupby(["title_role_bucket", "early_career_tier"])["salary_mid"]
    .agg(["median", "count"])
    .round(0)
    .to_string()
)


=== SALARY by ROLE (median, where disclosed) ===
                      median    n       min        max       std
title_role_bucket                                               
Analytics Engineer  136000.0   18  104000.0   445000.0   76494.0
Data Analyst         93235.0  105   41600.0   167500.0   28113.0
Data Engineer       131525.0   64   55000.0  1100000.0  127957.0
Data Scientist      142707.0   82       2.0   450000.0   62936.0

=== SALARY by ROLE × EARLY CAREER TIER ===
                                        median  count
title_role_bucket  early_career_tier                 
Analytics Engineer entry_or_junior    165852.0      1
                   mid                133750.0     12
Data Analyst       entry_or_junior     92500.0     17
                   mid                101000.0     45
Data Engineer      entry_or_junior    108000.0      9
                   mid                132500.0     33
Data Scientist     entry_or_junior     97400.0     12
                   mid        

In [9]:
# ── Cell 9 — AI blindspot ─────────────────────────────────────────────────────
print("\n=== AI ACKNOWLEDGMENT (overall) ===")
print(f"Acknowledges AI: {df['acknowledges_ai'].sum()} of {len(df)} ({df['acknowledges_ai'].mean():.0%})")
 
print("\n=== AI ACKNOWLEDGMENT by ROLE ===")
ai = (
    df.groupby("title_role_bucket")["acknowledges_ai"]
    .agg(["sum", "count", "mean"])
    .round(3)
)
ai.columns = ["yes", "total", "rate"]
print(ai.to_string())


=== AI ACKNOWLEDGMENT (overall) ===
Acknowledges AI: 226 of 448 (50%)

=== AI ACKNOWLEDGMENT by ROLE ===
                    yes  total   rate
title_role_bucket                    
Analytics Engineer   16     26  0.615
Data Analyst         55    196  0.281
Data Engineer        51    111  0.459
Data Scientist      104    115  0.904


In [10]:
# ── Cell 10 — title vs archetype confusion ────────────────────────────────────
print("\n=== TITLE vs LLM ARCHETYPE (confusion matrix, counts) ===")
matrix_df = df.dropna(subset=["role_archetype"]).copy()
pivot = (
    matrix_df.groupby(["title_role_bucket", "role_archetype"])
    .size()
    .unstack(fill_value=0)
)
print(pivot.to_string())
 
print("\n=== TITLE vs LLM ARCHETYPE (row %, agreement on diagonal) ===")
print(pivot.div(pivot.sum(axis=1), axis=0).round(2).to_string())
 
# Agreement rate — exact normalized match, not word-overlap.
# NOTE: the previous version of this check used set-overlap on underscore-
# split words (e.g. "analytics_engineer" vs "data_engineer" both contain
# "engineer"), which silently counted Analytics Engineer postings the LLM
# classified as Data Engineer as "agreement." That inflated the headline
# rate and masked the actual AE/DE confusion visible in the pivot table
# above. This version requires the normalized strings to match exactly.
def roles_match(row):
    title_bucket = row["title_role_bucket"].lower().replace(" ", "_").replace("-", "_")
    archetype = row["role_archetype"].lower()
    return title_bucket == archetype
 
agree_n = matrix_df.apply(roles_match, axis=1).sum()
print(f"\nAgreement: {agree_n} of {len(matrix_df)} ({agree_n/len(matrix_df):.0%})")


=== TITLE vs LLM ARCHETYPE (confusion matrix, counts) ===
role_archetype      analytics_engineer  data_analyst  data_engineer  data_scientist  hybrid
title_role_bucket                                                                          
Analytics Engineer                  22             0              2               0       2
Data Analyst                         1           186              1               3       5
Data Engineer                        0             1            108               0       2
Data Scientist                       0             0              1             113       1

=== TITLE vs LLM ARCHETYPE (row %, agreement on diagonal) ===
role_archetype      analytics_engineer  data_analyst  data_engineer  data_scientist  hybrid
title_role_bucket                                                                          
Analytics Engineer                0.85          0.00           0.08            0.00    0.08
Data Analyst                      0.01          0.

In [11]:
# ── Cell 11 — top skills overall and by role ──────────────────────────────────
print("\n=== TOP 20 REQUIRED SKILLS (overall) ===")
all_req = [t for row in df["tech_stack_required"] if isinstance(row, list) for t in row if isinstance(t, str)]
print(pd.Series(Counter(all_req)).sort_values(ascending=False).head(20).to_string())
 
print("\n=== TOP 10 REQUIRED SKILLS by ROLE ===")
for role in df["title_role_bucket"].unique():
    subset = df[df["title_role_bucket"] == role]
    tools = [t for row in subset["tech_stack_required"] if isinstance(row, list) for t in row if isinstance(t, str)]
    top = pd.Series(Counter(tools)).sort_values(ascending=False).head(10)
    print(f"\n--- {role} ---")
    print(top.to_string())


=== TOP 20 REQUIRED SKILLS (overall) ===
sql             291
python          248
excel            88
r                59
power bi         57
tableau          52
snowflake        50
aws              41
dbt              36
databricks       33
airflow          27
bigquery         26
git              25
pandas           23
pyspark          22
spark            21
looker           17
azure            16
numpy            16
scikit-learn     15

=== TOP 10 REQUIRED SKILLS by ROLE ===

--- Data Analyst ---
sql           114
excel          82
python         51
tableau        38
power bi       38
r              23
looker         15
word            9
snowflake       9
powerpoint      9

--- Data Scientist ---
python          100
sql              77
r                31
pandas           18
scikit-learn     14
numpy            14
snowflake        13
aws               9
databricks        8
git               8

--- Analytics Engineer ---
sql          24
python       11
dbt           9
snowflake     7


In [12]:
# ── Cell 12 — top paradigms by role ──────────────────────────────────────────
print("\n=== TOP 10 PARADIGMS by ROLE ===")
for role in df["title_role_bucket"].unique():
    subset = df[df["title_role_bucket"] == role]
    paras = [
        t
        for req, pref in zip(subset["paradigms_required"], subset["paradigms_preferred"])
        for row in [req, pref] if isinstance(row, list)
        for t in row if isinstance(t, str)
    ]
    top = pd.Series(Counter(paras)).sort_values(ascending=False).head(10)
    print(f"\n--- {role} ---")
    print(top.to_string())


=== TOP 10 PARADIGMS by ROLE ===

--- Data Analyst ---
data analysis            68
data quality             55
data visualization       52
data governance          37
statistical analysis     28
data modeling            28
data validation          25
data management          19
business intelligence    13
data cleaning            12

--- Data Scientist ---
machine learning        51
statistical analysis    36
data analysis           25
predictive modeling     19
causal inference        17
experimental design     16
data modeling           15
nlp                     13
data quality            12
statistical modeling    10

--- Analytics Engineer ---
data modeling           19
etl design              10
data governance         10
data quality             9
ci/cd                    4
data testing             2
machine learning         2
statistical analysis     2
self-serve analytics     2
data architecture        2

--- Data Engineer ---
etl design                62
data quality        

In [13]:
# ── Cell 13 — experience requirements ────────────────────────────────────────
print("\n=== YEARS REQUIRED by ROLE (median, where specified) ===")
yrs = df.dropna(subset=["years_required_min"])
print(
    yrs.groupby("title_role_bucket")["years_required_min"]
    .agg(["median", "count"])
    .round(1)
    .to_string()
)

print("\n=== YEARS REQUIRED by ROLE × EARLY CAREER TIER ===")
yrs_tier = yrs[yrs["early_career_tier"].isin(EARLY_CAREER_ORDER)]
print(
    yrs_tier.groupby(["title_role_bucket", "early_career_tier"])["years_required_min"]
    .agg(["median", "count"])
    .round(1)
    .to_string()
)


=== YEARS REQUIRED by ROLE (median, where specified) ===
                    median  count
title_role_bucket                
Analytics Engineer     4.0     25
Data Analyst           2.0    168
Data Engineer          3.0    104
Data Scientist         3.0    106

=== YEARS REQUIRED by ROLE × EARLY CAREER TIER ===
                                      median  count
title_role_bucket  early_career_tier               
Analytics Engineer entry_or_junior       2.0      1
                   mid                   4.0     15
Data Analyst       entry_or_junior       2.0     21
                   mid                   3.0     78
Data Engineer      entry_or_junior       0.5     14
                   mid                   3.0     53
Data Scientist     entry_or_junior       2.0     18
                   mid                   3.0     63


In [14]:
# ── Cell 14 — degree requirements ────────────────────────────────────────────
print("\n=== DEGREE REQUIREMENTS by ROLE ===")
deg = df.dropna(subset=["degree_requirement"])
print(
    deg.groupby(["title_role_bucket", "degree_requirement"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)


=== DEGREE REQUIREMENTS by ROLE ===
degree_requirement  bachelors  equivalent_ok  masters  none
title_role_bucket                                          
Analytics Engineer         11              2        0    13
Data Analyst              102             18        8    68
Data Engineer              42              3        6    60
Data Scientist             44              6       29    36


In [15]:
# ── Cell 15 — encourages applicants ──────────────────────────────────────────
print("\n=== ENCOURAGES APPLICANTS (overall) ===")
print(f"Yes: {df['explicitly_encourages_applicants'].sum()} of {len(df)} ({df['explicitly_encourages_applicants'].mean():.1%})")

print("\n=== ENCOURAGES APPLICANTS by ROLE ===")
enc = (
    df.groupby("title_role_bucket")["explicitly_encourages_applicants"]
    .agg(["sum", "count", "mean"])
    .round(3)
)
enc.columns = ["yes", "total", "rate"]
print(enc.to_string())


=== ENCOURAGES APPLICANTS (overall) ===
Yes: 52 of 448 (11.6%)

=== ENCOURAGES APPLICANTS by ROLE ===
                    yes  total   rate
title_role_bucket                    
Analytics Engineer    1     26  0.038
Data Analyst         22    196  0.112
Data Engineer         7    111  0.063
Data Scientist       22    115  0.191


In [16]:
# ── listed vs. inferred seniority mismatch ────────────────────────
print("\n=== LISTED vs INFERRED SENIORITY (counts) ===")
mismatch_df = df.dropna(subset=["listed_seniority", "inferred_seniority"]).copy()
pivot = (
    mismatch_df.groupby(["listed_seniority", "inferred_seniority"])
    .size()
    .unstack(fill_value=0)
)
print(pivot.to_string())

print("\n=== LISTED vs INFERRED SENIORITY (row %, agreement on diagonal) ===")
print(pivot.div(pivot.sum(axis=1), axis=0).round(2).to_string())

print(f"\nTotal rows with both fields populated: {len(mismatch_df)}")

print("\n=== YEARS REQUIRED by LISTED SENIORITY (overlap check) ===")
yrs_listed = df.dropna(subset=["years_required_min", "listed_seniority"])
print(
    yrs_listed.groupby("listed_seniority")["years_required_min"]
    .agg(["median", "min", "max", "count"])
    .round(1)
    .to_string()
)


=== LISTED vs INFERRED SENIORITY (counts) ===
inferred_seniority  entry  junior  mid  senior
listed_seniority                              
entry_level            12       0    0       0
junior                 12      43    1       0
mid_level              19      49  119      38

=== LISTED vs INFERRED SENIORITY (row %, agreement on diagonal) ===
inferred_seniority  entry  junior   mid  senior
listed_seniority                               
entry_level          1.00    0.00  0.00    0.00
junior               0.21    0.77  0.02    0.00
mid_level            0.08    0.22  0.53    0.17

Total rows with both fields populated: 293

=== YEARS REQUIRED by LISTED SENIORITY (overlap check) ===
                  median  min   max  count
listed_seniority                          
entry_level          0.0  0.0   0.0      3
junior               2.0  0.0   3.0     51
mid_level            3.0  0.0  10.0    209
